# M4 · The Agent Harness ★

> **The centerpiece of the workshop.** Everything so far — tools, sessions,
> memory, compaction — gets assembled into one **batteries-included** agent.
>
> **You'll use:** `create_harness_agent(...)`, and every battery it ships with.

---

As you build real agents you keep re-assembling the same machinery: the tool
loop, history persistence, compaction, planning, memory, approvals, observability.
The **agent harness** is that machinery, packaged.

![Agent harness](../../assets/agent-harness.png)

| Battery | What it adds | Built by hand in… |
|:--|:--|:--|
| **Function invocation** | the automatic tool-calling loop | M2 |
| **History + persistence** | conversation saved after *every model call* | M3 |
| **Compaction** | automatic context-window management | M3 (concept) |
| **TodoProvider** | the agent plans & tracks its own work items | *new* |
| **AgentModeProvider** | plan vs. execute mode tracking | *new* |
| **FileMemoryProvider** | durable file-based memory that survives compaction | M3 (concept) |
| **FileAccessProvider** | real read/write access to a folder | *new* |
| **Tool approval** | human-in-the-loop gating + "don't ask again" rules | *new* |
| **SkillsProvider** | progressive discovery/loading of skills | M3 |
| **BackgroundAgentsProvider** | delegate work to sub-agents concurrently | *new* |
| **Looping** | keep re-invoking until the plan is done | *new* |
| **Web search** | real-time web search tool | *new* |
| **OpenTelemetry** | built-in tracing | M7 |

**This notebook is self-contained.** Every battery below is a cell you can run,
and §15 rebuilds **Plan my next day at Cypher** end to end — plan, validate,
autonomous execution loop, and verified Excel agenda — without leaving Jupyter.

[Download this notebook](https://monuminu.github.io/Cypher-workshop26/modules/04-agent-harness/04-agent-harness.ipynb)

!!! warning "Version & experimental features"
    The harness ships in Agent Framework **core 1.13.0** (pinned by this
    workshop's `pyproject.toml`). `create_harness_agent` itself is a **stable**
    API, but a few batteries it can wire in — file access, background agents,
    looping — are still marked *experimental* and emit warnings. Fine for
    learning; pin versions in production. If `create_harness_agent` is missing,
    your install predates core 1.12.0; re-run [Setup](../setup.md).

## 1. Setup

One cell of shared scaffolding used by the whole lab:

- `HARNESS` — the token budgets and chat options every harness agent here gets.
- `render(...)` — a tiny streaming printer so you can *watch* the agent think,
  call tools, and search. Streaming emits one chunk per token, so tool calls are
  de-duplicated on `call_id` — otherwise a single file write prints 30+ lines.
- `WORK` — a scratch folder for everything the agent writes to disk.

About `default_options={"store": False}`: some providers (the OpenAI Responses
API among them) keep conversation state **server-side** by default. The harness
wants the *local* history to be authoritative — that's the copy its compaction
and persistence operate on. `store=False` says "don't keep it on the server, keep
it here." It's a standard chat option, so this works on every provider.

In [ ]:
import pathlib
import shutil
import sys
import warnings

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "workshop_utils").is_dir())
sys.path.insert(0, str(REPO))
warnings.filterwarnings("ignore")  # silence experimental-feature warnings for the lab

from workshop_utils import get_chat_client

WORK = pathlib.Path.cwd() / ".harness" / "m4"  # scratch root (gitignored)
shutil.rmtree(WORK, ignore_errors=True)  # start every run from a clean slate
WORK.mkdir(parents=True, exist_ok=True)

# Budgets + options shared by every harness agent in this notebook.
HARNESS = {
    "max_context_window_tokens": 128_000,
    "max_output_tokens": 4_096,
    "default_options": {"store": False},
}


async def render(agent, prompt, session, *, show_tools=True):
    """Stream one run: print the text, and one line per distinct tool call."""
    seen = set()
    async for update in agent.run(prompt, session=session, stream=True):
        for content in update.contents:
            kind = getattr(content, "type", "")
            if kind == "function_call" and show_tools:
                call_id = getattr(content, "call_id", None)
                if call_id not in seen:
                    seen.add(call_id)
                    print(f"\n  [tool: {content.name}]", flush=True)
            elif kind == "search_tool_call" and show_tools:
                print("\n  [web search]", flush=True)
            elif kind == "text":
                print(content.text, end="", flush=True)
    print()


print("provider client :", type(get_chat_client()).__name__)
print("repo root       :", REPO)
print("scratch folder  :", WORK)

## 2. Anatomy: what `create_harness_agent` actually wires

The factory needs only a **client**. The two token budgets switch on compaction.
Everything else is defaults.

Rather than take the table above on trust, **print the assembled agent**. Those
`context_providers` and `middleware` lists are the batteries, in the order they
run.

In [ ]:
from agent_framework import create_harness_agent

agent = create_harness_agent(
    client=get_chat_client(),
    name="HarnessAgent",
    description="A batteries-included assistant that plans and tracks its work.",
    **HARNESS,
)

print("context providers (run in this order, before every model call)")
for provider in agent.context_providers:
    print(f"   - {type(provider).__name__:<24} source_id={provider.source_id!r}")

print("\nmiddleware (outermost first)")
for mw in agent.middleware or []:
    print(f"   - {type(mw).__name__}")

print("\ntools auto-attached")
for tool in agent.default_options.get("tools") or []:
    print(f"   - {tool}")

print(f"\ncompaction strategy : {type(agent.compaction_strategy).__name__}")
print(f"telemetry source    : {agent.otel_provider_name}")
print(f"max_tokens          : {agent.default_options['max_tokens']}")

Read that output against the batteries table:

| Printed | Battery |
|:--|:--|
| `InMemoryHistoryProvider` | conversation history (M3) |
| `CompactionProvider` + `ContextWindowCompactionStrategy` | context-window management |
| `TodoProvider` | planning & work tracking |
| `AgentModeProvider` | plan/execute modes |
| `FileMemoryProvider` | durable memory on disk |
| `ToolApprovalMiddleware` | human-in-the-loop gating |
| `MessageInjectionMiddleware` | lets a host push messages mid-run |
| `{'type': 'web_search'}` | the built-in web search tool |

The harness also sets `require_per_service_call_history_persistence=True` — history
is saved after **every model call**, not just at the end of a turn. That's what makes
a long multi-tool run crash-safe.

Now let's take each battery in turn.

## 3. Battery — `TodoProvider`: the agent plans its own work

The provider injects five tools (`todos_add`, `todos_complete`, `todos_remove`,
`todos_get_remaining`, `todos_get_all`) and re-injects the **current todo list** as
a message before every model call. So the agent always sees its own plan.

Todo state lives in the **session**, not the agent — which means you can read it
back out and render it however you like.

In [ ]:
from agent_framework import TodoProvider


def provider_of(agent, provider_type):
    """Pull one of the harness's built-in context providers off the agent."""
    return next(p for p in agent.context_providers if isinstance(p, provider_type))


async def show_todos(agent, session):
    todo = provider_of(agent, TodoProvider)
    items = await todo.store.load_items(session, source_id=todo.source_id)
    if not items:
        print("  (no todos yet)")
        return items
    for item in items:
        mark = "x" if item.is_complete else " "
        print(f"  [{mark}] {item.id}. {item.title}")
    return items


session = agent.create_session()

await render(
    agent,
    "I want to launch a personal blog. Use your todo tools to record a short plan "
    "of 3 concrete steps, then briefly summarise it. Don't do the work yet.",
    session,
)

print("\n--- todo state read back from the session ---")
_ = await show_todos(agent, session)

The agent called `todos_add` and the items are now durable session state. You never
wrote a planner — the provider supplied the tools and the model used them.

## 4. Battery — `AgentModeProvider`: plan vs. execute

The harness ships two modes, and they change the agent's *whole operating procedure*:

| Mode | Behaviour |
|:--|:--|
| **plan** (default) | Interactive. Analyse, build a todo list, **ask clarifying questions**, get approval before doing work. |
| **execute** | Autonomous. Make reasonable decisions alone, don't ask, mark todos complete, keep going until done. |

The agent can switch modes itself with the `mode_set` tool, and *you* can switch it
from the host with `set_agent_mode(...)`. Reading it is `get_agent_mode(...)`.

In [ ]:
from agent_framework import AgentModeProvider, get_agent_mode, set_agent_mode

mode = provider_of(agent, AgentModeProvider)
print("available modes :", list(mode.available_modes))
print("default mode    :", mode.default_mode)
print("this session    :", get_agent_mode(session))

# The host can drive the mode directly — this is the hand-off used in section 15.
set_agent_mode(session, "execute")
print("after set_agent_mode:", get_agent_mode(session))

set_agent_mode(session, "plan")  # put it back for the rest of this section
print("restored            :", get_agent_mode(session))

Switching modes externally also injects a **"mode changed"** notice into the next
turn. That matters: system instructions alone are often not enough to redirect a
model that has already seen its own `mode_set` call earlier in the history.

## 5. Battery — `FileMemoryProvider`: memory that survives compaction

In M3 you learned that context is finite and compaction eventually throws things
away. File memory is the escape hatch: the agent writes what matters to **real files**,
and gets back an index of them on every turn.

Tools: `file_memory_write`, `file_memory_read`, `file_memory_ls`, `file_memory_grep`,
`file_memory_replace`, `file_memory_replace_lines`, `file_memory_delete`.

By default the store is `{cwd}/agent-file-memory`, scoped **per session**. Here we
point it at our scratch folder so it's easy to inspect.

In [ ]:
from agent_framework import FileSystemAgentFileStore

MEMORY_DIR = WORK / "memory"

memo_agent = create_harness_agent(
    client=get_chat_client(),
    name="MemoAgent",
    file_memory_store=FileSystemAgentFileStore(MEMORY_DIR),
    disable_web_search=True,
    **HARNESS,
)
memo_session = memo_agent.create_session()

await render(
    memo_agent,
    "Remember this for later: our production database is PostgreSQL 16 and it is "
    "deployed in eastus2. Save it to a memory file called infra.md.",
    memo_session,
)

print("\n--- files on disk ---")
for path in sorted(MEMORY_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(MEMORY_DIR)}  ({path.stat().st_size} bytes)")

In [ ]:
# A later turn — the agent finds the memory through its index, not the chat history.
await render(memo_agent, "Which region is our production database in?", memo_session)

Note the extra files the provider maintains for you: a `memories.md` **index** and a
`*_description.md` companion per memory. The index is injected each turn, so the agent
knows what it has stored without reading every file — the same progressive-disclosure
idea as skills.

## 6. Battery — web search

If the client implements `SupportsWebSearchTool`, the harness attaches a web search
tool automatically. That's what turns a harness agent into a *researcher*.

In [ ]:
search_agent = create_harness_agent(
    client=get_chat_client(),
    name="SearchAgent",
    agent_instructions="Answer in two sentences and cite your source inline as [name](url).",
    disable_file_memory=True,
    **HARNESS,
)

await render(
    search_agent,
    "What is the Model Context Protocol? Search the web and cite a source.",
    search_agent.create_session(),
)

!!! note "If your provider has no web search"
    You'll see a warning at construction time and the tool simply won't be attached.
    Pass `disable_web_search=True` to silence it — every other battery still works,
    and §15 will fall back to the model's own knowledge.

## 7. Battery — your own tools alongside the batteries

`tools=[...]` takes exactly the function tools you wrote in M2. This is the
"meet your claw" pattern: harness machinery **plus** your domain functions.

In [ ]:
from typing import Annotated


def get_stock_price(
    ticker: Annotated[str, "Stock ticker symbol, e.g. MSFT"],
) -> str:
    """Get the current share price for a ticker symbol."""
    prices = {"MSFT": 512.43, "AAPL": 271.10, "NVDA": 187.62}
    price = prices.get(ticker.upper())
    return f"{ticker.upper()}: ${price}" if price else f"No price on file for {ticker!r}."


claw = create_harness_agent(
    client=get_chat_client(),
    name="Claw",
    agent_instructions="You are a personal finance assistant. Be concise.",
    tools=[get_stock_price],
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)

await render(
    claw,
    "What would 10 shares each of MSFT and NVDA cost me in total?",
    claw.create_session(),
)

## 8. Battery — `FileAccessProvider`: give the agent a folder

File *memory* is the agent's private scratchpad. File *access* is a **shared**
folder you point it at — your data, your outputs. This is the .NET
`Harness_Step03_DataProcessing` scenario.

It's opt-in: pass `file_access_store=...`. Tools: `file_access_read`, `ls`, `grep`,
`write`, `delete`, `replace`, `replace_lines`.

In [ ]:
DATA_DIR = WORK / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "sales.csv").write_text(
    "region,quarter,revenue\n"
    "emea,Q1,120000\n"
    "emea,Q2,138000\n"
    "amer,Q1,210000\n"
    "amer,Q2,199000\n",
    encoding="utf-8",
)

data_agent = create_harness_agent(
    client=get_chat_client(),
    name="DataAgent",
    agent_instructions="You analyse CSV data files. Be concise.",
    file_access_store=FileSystemAgentFileStore(DATA_DIR),
    # Approvals are covered in section 9 — turn them off here so the cell runs unattended.
    file_access_disable_readonly_tool_approval=True,
    file_access_disable_write_tool_approval=True,
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)

await render(
    data_agent,
    "Read sales.csv, work out total revenue per region, and write the answer to summary.md.",
    data_agent.create_session(),
)

print("\n--- files on disk ---")
for path in sorted(DATA_DIR.iterdir()):
    print(f"  {path.name}")
print("\n--- summary.md ---")
print((DATA_DIR / "summary.md").read_text(encoding="utf-8"))

## 9. Battery — tool approval

Notice we *disabled* approvals above. By default, file-access write tools **require
human approval**: the run stops and hands you back a `function_approval_request`
instead of doing the thing.

The harness's built-in tools (`todos_*`, `file_memory_*`) register as
`never_require`, so a plain harness agent never interrupts you. Approval exists for
the batteries that touch the outside world.

First, watch a run get gated:

In [ ]:
GATED_DIR = WORK / "gated"
GATED_DIR.mkdir(parents=True, exist_ok=True)

gated_agent = create_harness_agent(
    client=get_chat_client(),
    name="GatedAgent",
    file_access_store=FileSystemAgentFileStore(GATED_DIR),  # approvals left ON
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)

result = await gated_agent.run(
    "Write a file called report.txt containing the text 'quarterly numbers'.",
    session=gated_agent.create_session(),
)

pending = [
    c
    for message in result.messages
    for c in message.contents
    if getattr(c, "type", "") == "function_approval_request"
]
print("assistant text :", repr(result.text))
print("pending approvals:", [c.function_call.name for c in pending])
print("files written  :", [p.name for p in GATED_DIR.iterdir()] or "none — the write was gated")

Nothing was written. The agent is *waiting on you*.

An interactive host answers by sending the approval back:

```python
from agent_framework import Message, create_always_approve_tool_response

responses = [create_always_approve_tool_response(req, reason="user clicked Allow") for req in pending]
await agent.run(Message(role="user", contents=responses), session=session)
```

`create_always_approve_tool_response` also records a **standing rule** — the
"don't ask again for this tool" checkbox — so subsequent calls sail through.

!!! warning "Provider caveat"
    Replaying a *local* tool's approval request is not supported by every provider.
    The OpenAI Responses client serialises it as an MCP approval request and the
    service rejects it (`Expected an ID that begins with 'mcpr'`). Which is why this
    notebook demonstrates the second mechanism instead — **auto-approval rules** —
    which work everywhere and are what you'd want for a non-interactive job anyway.

`auto_approval_rules` are predicates over the pending call. Return `True` and the
harness approves it without asking. Here: allow `.txt` writes, gate everything else.

In [ ]:
import json


def approve_txt_writes(function_call) -> bool:
    """Auto-approve writes to .txt files; anything else still needs a human."""
    raw = function_call.arguments or "{}"
    args = json.loads(raw) if isinstance(raw, str) else raw
    return str(args.get("file_name", "")).endswith(".txt")


auto_agent = create_harness_agent(
    client=get_chat_client(),
    name="AutoApproveAgent",
    file_access_store=FileSystemAgentFileStore(GATED_DIR),
    auto_approval_rules=[approve_txt_writes],
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)
auto_session = auto_agent.create_session()

await render(
    auto_agent,
    "Write a file called report.txt containing the text 'quarterly numbers'.",
    auto_session,
)
print("files written:", [p.name for p in GATED_DIR.iterdir()])

In [ ]:
# ...and a write the rule does NOT cover is still held for a human.
blocked = await auto_agent.run(
    "Now write a file called report.md containing the text 'markdown version'.",
    session=auto_agent.create_session(),
)
still_pending = [
    c
    for message in blocked.messages
    for c in message.contents
    if getattr(c, "type", "") == "function_approval_request"
]
print("still pending :", [c.function_call.name for c in still_pending])
print("files on disk :", sorted(p.name for p in GATED_DIR.iterdir()))

## 10. Battery — skills

M3 introduced skills: expertise on disk, loaded only when relevant. The harness
takes a folder — `skills_paths=[REPO / "skills"]` — and discovers every `SKILL.md`
under it. This repo already ships one at `skills/unit-converter/`.

That skill also ships a *script*, and running a script needs two extra things the
plain `skills_paths` shortcut can't give you (both familiar from M3):

- a **`script_runner`**, which decides *how* a script executes — the framework
  deliberately refuses to shell out on your behalf;
- an **approval rule**, because skill tools are gated by default.

So we build the provider by hand and pass it as `skills_provider=...`. The rest of
the harness is unchanged.

In [ ]:
import subprocess

from agent_framework import FileSkill, FileSkillScript, FileSkillsSource, SkillsProvider


def run_script(skill: FileSkill, script: FileSkillScript, args: list[str] | None = None) -> str:
    """Execute a file skill's script as a local Python subprocess (same helper as M3)."""
    done = subprocess.run(
        [sys.executable, str(script.full_path), *(args or [])],
        capture_output=True,
        text=True,
        timeout=30,
    )
    return (done.stdout + done.stderr).strip() or "(no output)"


skilled = create_harness_agent(
    client=get_chat_client(),
    name="SkilledAgent",
    skills_provider=SkillsProvider(
        FileSkillsSource(str(REPO / "skills"), script_runner=run_script)
    ),
    auto_approval_rules=[SkillsProvider.all_tools_auto_approval_rule],
    disable_web_search=True,
    disable_file_memory=True,
    disable_todo=True,
    **HARNESS,
)
print("providers:", [type(p).__name__ for p in skilled.context_providers])

# execute mode, so it acts on the skill instead of proposing to.
skilled_session = skilled.create_session()
set_agent_mode(skilled_session, "execute")

await render(
    skilled,
    "What skills do you have available? Then use one to convert 26.2 miles to kilometres.",
    skilled_session,
)

Notice the tool sequence: `load_skill` → `read_skill_resource` → `run_skill_script`.
The model only pulled the skill's full instructions once it decided the skill was
relevant — the catalog it sees up front is one line per skill. That is *progressive
disclosure*, and it's why skills scale where a giant system prompt doesn't.

## 11. Battery — background agents

`background_agents=[...]` lets the harness agent **delegate**. It gets tools to start
a task on a named sub-agent, poll or wait for it, and collect results — so slow work
runs concurrently instead of blocking the main conversation. This is the .NET
`Harness_Step02` scenario.

!!! warning "Trust boundary"
    Delegated agents receive text from this agent and their output flows back into its
    context. Only pass agents you've vetted.

In [ ]:
from agent_framework import Agent

quote_agent = Agent(
    client=get_chat_client(),
    name="QuoteAgent",
    description="Looks up share prices for a ticker symbol.",
    instructions="You report share prices. Answer with just the ticker and price.",
    tools=[get_stock_price],
)

delegator = create_harness_agent(
    client=get_chat_client(),
    name="Delegator",
    agent_instructions="Delegate price lookups to background agents, then summarise.",
    background_agents=[quote_agent],
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)
print("providers:", [type(p).__name__ for p in delegator.context_providers])

await render(
    delegator,
    "Use a background agent to look up the price of AAPL, wait for it, and tell me the result.",
    delegator.create_session(),
)

## 12. Battery — compaction

The two numbers you've been passing all along:

```python
max_context_window_tokens=128_000   # how big the model's window is
max_output_tokens=4_096             # how much you'll let it write per response
```

From those the harness builds a `ContextWindowCompactionStrategy` and runs it in
**two places**:

- **before** each model call — trims the outgoing messages so the request fits
- **after** each turn — compacts the persisted history in place

Omit both numbers and compaction is silently **off** — the harness has no budget to
reason about. That's the single easiest harness mistake to make.

In [ ]:
budgeted = create_harness_agent(client=get_chat_client(), **HARNESS)
unbudgeted = create_harness_agent(client=get_chat_client(), default_options={"store": False})

print("with budgets    :", type(budgeted.compaction_strategy).__name__)
print("  providers     :", [type(p).__name__ for p in budgeted.context_providers])
print()
print("without budgets :", budgeted.compaction_strategy is not None, "->", unbudgeted.compaction_strategy)
print("  providers     :", [type(p).__name__ for p in unbudgeted.context_providers])
print("\nNote the missing CompactionProvider in the unbudgeted agent.")

## 13. Battery — looping until the plan is done

One `agent.run(...)` is one turn. A plan with five todos needs five turns — unless
you loop.

`loop_should_continue` is a predicate checked after every iteration. The harness
ships the one you almost always want:

- **`todos_remaining(looping_modes=["execute"])`** — keep going while any todo is
  open *and* the agent is in execute mode. Planning stays interactive; execution runs
  to completion. (This is the Python counterpart of .NET's
  `TodoCompletionLoopEvaluator`.)
- **`todos_remaining_message`** — between iterations, remind the agent which items
  are still open.
- **`loop_max_iterations`** — the safety net.

Watch it drain a todo list on its own:

In [ ]:
from agent_framework import todos_remaining, todos_remaining_message

looper = create_harness_agent(
    client=get_chat_client(),
    name="Looper",
    agent_instructions=(
        "You answer short trivia questions. Work through your todo list one item at a "
        "time, answering each in a single sentence, and call todos_complete for each "
        "item as soon as you have answered it."
    ),
    loop_should_continue=todos_remaining(looping_modes=["execute"]),
    loop_next_message=todos_remaining_message,
    loop_max_iterations=6,
    disable_web_search=True,
    disable_file_memory=True,
    **HARNESS,
)
loop_session = looper.create_session()

# Seed a plan, then hand off to execute mode so the loop engages.
await render(
    looper,
    "Use todos_add to record exactly three todos: "
    "(1) name the tallest mountain, (2) name the longest river, (3) name the largest ocean. "
    "Do not answer them yet.",
    loop_session,
)
print("\n--- plan ---")
await show_todos(looper, loop_session)

In [ ]:
set_agent_mode(loop_session, "execute")
await render(looper, "Approved — work through every todo now.", loop_session)

print("\n--- after the loop ---")
items = await show_todos(looper, loop_session)
print(f"\nopen items remaining: {sum(1 for i in items if not i.is_complete)}")

You sent **one** message and the agent took several turns by itself, stopping the
moment the predicate went false. That's the whole autonomy story.

## 14. Battery — observability

Every harness agent is instrumented out of the box and reports under a dedicated
telemetry source, so harness traffic is distinguishable from hand-built agents.

In [ ]:
print("default source :", agent.otel_provider_name)

custom = create_harness_agent(
    client=get_chat_client(),
    otel_provider_name="cypher.workshop.m4",
    **HARNESS,
)
print("custom source  :", custom.otel_provider_name)

Point an OTLP exporter at it and you get spans for every model call, tool call, and
compaction pass for free. **M7 · Operationalizing** wires this to a real backend.

## 15. Plan my next day at Cypher, end to end

Rebuild the [opening demo](00-cypher-agenda-demo.ipynb) using the batteries above.
The business task and domain tools are identical for both agents. Inspect the
harness completion predicate: it combines task state with independent workbook
validation. Do not infer reliability from a single run or require the baseline to fail.

### Configure the participant · 1 minute

The same model, task, profile, captured schedule and four application tools go to
both agents. A basic `Agent` already has a tool-calling loop and can succeed.
Only the harness receives its runtime providers and completion-driven continuation.

Run **Setup → Cypher agenda demo** first. Set `RUN_AGENTS = True` deliberately to
make model calls. Default limits are 120 seconds and 16 model calls per initial run;
60 seconds and the same call cap per revision. Provider-side requests may finish
after cancellation; these limits are not a monetary spending guarantee.

In [ ]:
import json
import pathlib
import sys
import uuid
from datetime import datetime

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "workshop_utils").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from IPython.display import HTML, FileLink, display
from workshop_utils import get_chat_client
from workshop_utils.agenda_source import fetch_live_schedule, import_reviewed_schedule, next_day, IST
from workshop_utils.agenda import Profile, AgendaTools, TASK, verify_workbook
from workshop_utils.agenda_runtime import RunMeter, CompletionGate, HARNESS_POLICY, run_demo_agent, comparison

RUN_AGENTS = False  # Explicitly opt in to model calls after reviewing your provider configuration.
ATTENDANCE_DAY = None  # Next conference day; use an explicit YYYY-MM-DD for a rehearsal.
PASS_TYPE = "learning"  # Prepared persona; edit to match the participant's actual pass.
MANUAL_TSV = None  # Optional Path to freshly transcribed, reviewed official schedule rows.
CAPTURED_AT = None  # Actual ISO capture timestamp with +05:30, required for manual import.
MANUAL_REVIEWED = False
MAX_MODEL_CALLS = 16
INITIAL_SECONDS, REVISION_SECONDS = 120, 60
RUN = pathlib.Path.cwd() / ".harness" / "agenda" / uuid.uuid4().hex[:10]
RUN.mkdir(parents=True)
print("Artifacts:", RUN)

### Capture the official schedule · before the timed presentation

Fetch every conference day from the rendered official site. Both agents use this
one observation, including for the changed-preference round. A new comparison
fetches again; no previous run is silently reused.

If access fails, use the **fresh manual import** instructions in Setup. Inspect the
review table against the official source; PDF column order is never guessed.
Uncertain/excluded source records and provisional status remain visible.

In [ ]:
if MANUAL_TSV is None:
    schedule = await fetch_live_schedule()
else:
    schedule = import_reviewed_schedule(pathlib.Path(MANUAL_TSV), captured_at=CAPTURED_AT,
                                        reviewed=MANUAL_REVIEWED)
schedule.save(RUN / "schedule.json")
day = next_day(schedule, ATTENDANCE_DAY)
profile = Profile(day=day, pass_type=PASS_TYPE,
                  interests=("production agents", "evaluation", "governance"),
                  available=("09:00", "18:00"), lunch=("13:00", "13:30"),
                  unavailable=(), hall_buffer_minutes=5, must_attend=())
print("Source:", schedule.source_url, "\nCaptured:", schedule.captured_at)
print("Method:", schedule.method, "| Provisional:", schedule.provisional)
print("Fingerprint:", schedule.fingerprint, "| Sessions:", len(schedule.sessions))
print("Warnings:", schedule.warnings)
print("Participant:", profile)
display(HTML("<p><b>Would you trust this agenda enough to follow it tomorrow?</b></p>"))

### Construct the comparison

These are the important lines. Shared tools retrieve the captured data, validate
constraints, write Excel, and reopen it. They are **custom application tools**.
The harness adds todos, modes, file memory, history, and a bounded loop. Its loop
condition checks the real workbook as well as unfinished tasks. Native web search
is disabled here so both agents use the same official schedule observation.

We enter execute mode directly for the short opener; the agent plans visibly using
todos. The earlier M4 battery exercise teaches the interactive plan/approve step.
Neither agent receives the other's work.

In [ ]:
from agent_framework import Agent, create_harness_agent, FileSystemAgentFileStore, set_agent_mode

client = get_chat_client()  # Same configured model for both; no dependency/model upgrades.
basic_tools = AgendaTools(schedule, profile, RUN / "basic")
harness_tools = AgendaTools(schedule, profile, RUN / "harness")
basic_meter, harness_meter = RunMeter(MAX_MODEL_CALLS), RunMeter(MAX_MODEL_CALLS)
gate = CompletionGate(harness_tools)
options = {"store": False, "max_tokens": 4096}

basic = Agent(client=client, name="BasicAgent", instructions=TASK,
              tools=basic_tools.tools, default_options=options,
              middleware=basic_meter.middleware())
harness = create_harness_agent(
    client=client, name="HarnessAgent", agent_instructions=TASK,
    harness_instructions=HARNESS_POLICY, tools=harness_tools.tools,
    default_options=options, max_context_window_tokens=128000, max_output_tokens=4096,
    disable_web_search=True,
    file_memory_store=FileSystemAgentFileStore(RUN / "harness" / "memory"),
    loop_should_continue=gate.should_continue, loop_next_message=gate.next_message,
    loop_max_iterations=4, middleware=harness_meter.middleware(),
)
basic_session, harness_session = basic.create_session(), harness.create_session()
set_agent_mode(harness_session, "execute")
results = []

### Basic agent · up to 2 minutes

Watch which tools it chooses and inspect the actual workbook. A successful
baseline is useful evidence; do not rerun it until it fails.

In [ ]:
if RUN_AGENTS:
    basic_result = await run_demo_agent(basic, basic_session, basic_tools, basic_meter,
                                        seconds=INITIAL_SECONDS)
    results.append(basic_result)
    if basic_tools.path.exists():
        display(FileLink(basic_tools.path.relative_to(pathlib.Path.cwd()).as_posix()))
else:
    print("Model calls disabled. Set RUN_AGENTS = True to rehearse or present.")

### Harness agent · up to 2 minutes

Look for task tracking, constraint checks, and completion checks. Repairs are
visible **when needed**, not scripted. A timeout or exhausted call budget remains
unfinished even if the agent says it succeeded.

In [ ]:
if RUN_AGENTS:
    harness_result = await run_demo_agent(harness, harness_session, harness_tools, harness_meter,
                                          seconds=INITIAL_SECONDS, harness=True)
    results.append(harness_result)
    print("Tasks:", harness_result["tasks"])
    if harness_tools.path.exists():
        display(FileLink(harness_tools.path.relative_to(pathlib.Path.cwd()).as_posix()))
    display(HTML(comparison(results)))

### Change the requirement · up to 2 minutes

“I’m unavailable from 14:00–15:00; prioritize evaluation and governance.”

Both agents retain their own conversation history and receive the same updated
profile. A new workbook filename prevents the previous revision from passing as
the new result. Ordinary conversation continuity alone does **not** prove durable memory.

In [ ]:
if RUN_AGENTS:
    revised_profile = profile.revised()
    for bundle in (basic_tools, harness_tools):
        bundle.revise(revised_profile)
    basic_revision = await run_demo_agent(basic, basic_session, basic_tools, basic_meter,
                                          seconds=REVISION_SECONDS, revision=True)
    harness_revision = await run_demo_agent(harness, harness_session, harness_tools, harness_meter,
                                            seconds=REVISION_SECONDS, revision=True, harness=True)
    results.extend([basic_revision, harness_revision])
    display(HTML(comparison([basic_revision, harness_revision])))
    for bundle in (basic_tools, harness_tools):
        if bundle.path.exists():
            display(FileLink(bundle.path.relative_to(pathlib.Path.cwd()).as_posix()))

### Reveal and discuss · 1–2 minutes

Open **My Agenda**, **Alternatives**, **Profile**, and **Checks & Sources**. Compare
selection reasons, conflicts, hall transitions, pass restrictions and exported
facts. Explain any time/call cost of verification. A completed workbook means its
structural checks passed; a human still judges preference relevance.

If both agents succeed, discuss the explicit completion policy, visible tasks and
restart behavior. If either fails, show the check or provider error honestly.
Infeasible constraints require a draft and explanation, never invented sessions.

The runtime persists checkpoints and profile files locally. That host persistence
is separate from the harness's file-memory tools. This short demo does not establish
that compaction, skills, or background agents improve agenda quality.

## Your turn: change the runtime, measure the result

1. **Disable completion checks.** Set `gate.enabled = False`, construct a fresh pair
   with a new `RUN`, and repeat with the same captured schedule and profile. Keep
   the final independent audit enabled. Compare actual outcomes; failure is not required.
2. **Disable todos.** Add `disable_todo=True` and retain workbook-driven completion.
   Notice that task visibility and artifact verification are separate capabilities.
3. **Create an infeasible profile.** Require two overlapping sessions. Keep the
   resource limits. Expect an honest draft or explanation, not fabricated times.
4. **Restart with persisted preferences.** After a run, save `RUN` below, restart the
   kernel, and execute the recovery cell. Inspect the restored session ID and profile
   without any model calls. Recreate the harness using the same memory directory and
   use `restored_session` for a follow-up; point `harness_tools` at `restored_profile`.
   The host checkpoint restores conversation/todo state; FileMemoryProvider stores
   its own memory files. These are distinct from merely continuing a live chat.
5. **Inspect file memory.** List the memory directory and show the preference file
   only if the agent actually wrote one. Do not claim durable agent memory from the
   host's `profile.json` alone.

The compact task need not trigger compaction, skills, or background agents.
Use the dedicated battery exercises to demonstrate those capabilities.

In [ ]:
# Safe recovery exercise: explicitly choose a completed run directory.
# This cell makes no model calls and does not fetch or reuse old schedule data for a new comparison.
import json
import pathlib
import sys
from agent_framework import AgentSession

REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "workshop_utils").is_dir())
sys.path.insert(0, str(REPO))
from workshop_utils.agenda import Profile

RECOVER_RUN = None  # Set pathlib.Path("...") to the RUN printed by your prior execution.
if RECOVER_RUN is not None:
    recovery = pathlib.Path(RECOVER_RUN) / "harness"
    restored_profile = Profile.load(recovery / "profile.json")
    restored_session = AgentSession.from_dict(json.loads((recovery / "session.json").read_text(encoding="utf-8")))
    print("Restored session:", restored_session.session_id)
    print("Restored profile:", restored_profile)
    print("Memory files:", [str(p.relative_to(recovery)) for p in (recovery / "memory").rglob("*") if p.is_file()])
else:
    print("Choose RECOVER_RUN to demonstrate a real restart.")

Continue to [M5 · Multi-Agent Orchestration](05-orchestration.ipynb).